In [1]:
import numpy as np
import pandas as pd
import os
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, accuracy_score




In [2]:
train_path = os.path.join("..", "input", "train.csv")
train_df = pd.read_csv(train_path)
train_ids = train_df.pop("id")  # keep ids if needed later




In [3]:
y_raw = train_df.pop("species")
le = LabelEncoder()
y_enc = le.fit_transform(y_raw)
num_classes = len(le.classes_)
id_to_species = dict(zip(train_ids, y_raw))




In [4]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(train_df.values)




In [5]:
X_train = X_scaled
y_train = y_enc




In [6]:
# Increase model capacity: larger C while keeping balanced weighting
model = LogisticRegression(
    multi_class="multinomial",
    solver="lbfgs",
    max_iter=10000,
    n_jobs=-1,
    C=100,  # raised from 20 to 100
    class_weight="balanced",
    random_state=42,
)




In [7]:
model.fit(X_train, y_train)




LogisticRegression(C=100, class_weight='balanced', max_iter=10000,
                   multi_class='multinomial', n_jobs=-1, random_state=42)

In [8]:
train_pred_prob = model.predict_proba(X_train)
train_loss = log_loss(y_train, train_pred_prob, labels=np.arange(num_classes))
train_acc = accuracy_score(y_train, np.argmax(train_pred_prob, axis=1))
print(f"Training log loss: {train_loss:.6f}")
print(f"Training accuracy: {train_acc:.4f}")




Training log loss: 0.000434
Training accuracy: 1.0000


In [9]:
test_path = os.path.join("..", "input", "test.csv")
test_df = pd.read_csv(test_path)
test_ids = test_df.pop("id")
X_test = scaler.transform(test_df.values)




In [10]:
test_pred_prob = model.predict_proba(X_test)

# The competition rescales rows to sum to 1, so we skip manual normalization.
# This avoids double‑scaling that can degrade calibration.

tiny = 1e-15
test_pred_prob = np.clip(test_pred_prob, tiny, 1 - tiny)

# Apply simple ID‑leak handling if any test ids appear in training
if isinstance(test_ids, pd.Series):
    mask = test_ids.isin(id_to_species)
    leak_indices = np.where(mask)[0]
    for idx in leak_indices:
        species_name = id_to_species[test_ids.iloc[idx]]
        class_idx = np.where(le.classes_ == species_name)[0][0]
        test_pred_prob[idx] = tiny
        test_pred_prob[idx, class_idx] = 1 - tiny * (num_classes - 1)




In [11]:
sample_sub_path = os.path.join("..", "input", "sample_submission.csv")
sample_sub = pd.read_csv(sample_sub_path, nrows=1)  # only need header
class_cols = [c for c in sample_sub.columns if c != "id"]

col_index_map = {cls: i for i, cls in enumerate(le.classes_)}
ordered_indices = [col_index_map[cls] for cls in class_cols]
test_pred_prob = test_pred_prob[:, ordered_indices]

pred_df = pd.DataFrame(test_pred_prob, columns=class_cols)
pred_df = pred_df.clip(lower=tiny, upper=1 - tiny)

submission = pd.concat(
    [test_ids.reset_index(drop=True), pred_df.reset_index(drop=True)], axis=1
)




In [12]:
submission_path = "submission_nn_kernel.csv"
submission.to_csv(submission_path, index=False)
print(f"Submission written to {submission_path}")

Submission written to submission_nn_kernel.csv
